In [28]:
import pandas as pd
import numpy as np
df = pd.read_csv('D:/Downloads/merge.csv', low_memory=False)

In [29]:
print(df.value_counts('Signature'))

Signature
APT    63125
AA     34109
Name: count, dtype: int64


In [22]:
list_features = [
    'bidirectional_stddev_ps',
    'bidirectional_mean_ps',
    'dst2src_duration_ms',
    'src2dst_duration_ms',
    'bidirectional_bytes',
    'bidirectional_packets',
    'bidirectional_duration_ms',
    'src2dst_packets',
    'dst2src_packets',
    'src2dst_bytes',
    'dst2src_bytes',
    'bidirectional_mean_piat_ms',
    'bidirectional_stddev_piat_ms',
    'bidirectional_max_piat_ms',
    'bidirectional_min_piat_ms',
    'src2dst_mean_piat_ms',
    'src2dst_stddev_piat_ms',
    'src2dst_max_piat_ms',
    'src2dst_min_piat_ms',
    'dst2src_mean_piat_ms',
    'dst2src_stddev_piat_ms',
    'dst2src_max_piat_ms',
    'dst2src_min_piat_ms',
    'bidirectional_fin_packets',
    'bidirectional_syn_packets',
    'bidirectional_rst_packets',
    'bidirectional_psh_packets',
    'bidirectional_ack_packets',
    'bidirectional_urg_packets',
    'bidirectional_cwr_packets',
    'bidirectional_ece_packets',
    'time',
    'locate',
    'Stage',
    'Activity'
]

# Lọc DataFrame theo list_features
df = df[list_features]

In [23]:
# Giả sử stage_mapping là từ điển mà bạn đã định nghĩa
action_mapping = {
    'Maintain Access': 0,
    'Encrypted Channel: Symmetric Cryptography': 1,     
    'Data Transfer Size Limits': 2,
    'Remote System Discovery': 3,
    'Exfiltration over C2 channel': 4,
    'Remove Traces': 5,
    'Unsecured Credentials': 6,
    'Active Scanning: Scanning IP Blocks': 7,
    'Active Scanning: Vulnerability Scanning': 8,
    'Bruteforce: Password Guessing': 9
}

stage_mapping = {
    'Reconnaissance': 0,     
    'Establish Foothold': 1,
    'Lateral Movement': 2,
    'Data Exfiltration': 3,
    'Cover up': 4
}

In [24]:
df['Activity'] = df['Activity'].map(action_mapping)
df['Stage'] = df['Stage'].map(stage_mapping)

In [25]:
# Hàm gán nhãn 'who' với kiểm tra thêm Activity
def assign_who(row):
    # Điều kiện cho nhãn AA
    if (row['locate'] in [0, 4]) and (row['Stage'] == 0) and (row['Activity'] in [7, 8, 9]):
        return 'AA'
    # Điều kiện cho nhãn APT
    elif (row['locate'] in [0, 3, 5]) and (row['Stage'] in [0, 1, 2, 3, 4]) and (row['Activity'] in [0, 1, 2, 3, 4, 5, 6]):
        return 'APT'
    # Trường hợp không thỏa mãn điều kiện nào
    else:
        return 'Unknown'

In [26]:
# Áp dụng hàm assign_who cho từng hàng
df['who'] = df.apply(assign_who, axis=1)

In [27]:
# Kiểm tra kết quả
print("Số lượng mỗi nhãn trong cột 'who':")
print(df['who'].value_counts())
print("\nXem trước 5 dòng đầu của DataFrame:")
print(df[['locate', 'Stage', 'who']].head())

Số lượng mỗi nhãn trong cột 'who':
who
APT    63125
AA     34109
Name: count, dtype: int64

Xem trước 5 dòng đầu của DataFrame:
   locate  Stage  who
0       3      2  APT
1       3      2  APT
2       3      2  APT
3       3      2  APT
4       3      2  APT


In [8]:
df.to_csv('D:/Downloads/APT_combined_label.csv', index=False)